In [2]:
!apt-get install -y zstd pciutils -qq

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import os

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(10)

!curl http://localhost:11434

Selecting previously unselected package pci.ids.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Setting up libpci3:amd64 (1:3.7.0-6) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Setting up pciutils (1:3.7.0-6) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbb

In [3]:
!ollama pull qwen2.5:7b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 2bada8a74506:   0% ▕                  ▏ 309 KB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   1% ▕                  ▏  37 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏  76 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   4% ▕                  ▏ 168 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   6% ▕█                 ▏ 261 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 306 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   8% ▕█                 ▏ 396 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  11% ▕█                 ▏ 493 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  11% 

In [4]:
!ollama list

]11;?\NAME          ID              SIZE      MODIFIED      
qwen2.5:7b    845dbda0ea48    4.7 GB    5 seconds ago    


In [5]:
!curl http://localhost:11434

Ollama is running

In [6]:
!pip install -U ollama

# calculations.py

In [7]:
from typing import Optional


DEFAULT_VAT_RATE = 0.14
DEFAULT_TOLERANCE = 0.05


# ============================================================
# BASIC CALCULATIONS
# ============================================================

def calculate_net_amount(subtotal: float, discount: float = 0.0) -> float:
    if subtotal < 0:
        raise ValueError("Subtotal cannot be negative.")

    if discount < 0:
        raise ValueError("Discount cannot be negative.")

    if discount > subtotal:
        raise ValueError("Discount cannot exceed subtotal.")

    return subtotal - discount


def calculate_vat(taxable_amount: float, vat_rate: Optional[float] = None) -> float:
    if taxable_amount < 0:
        raise ValueError("Taxable amount cannot be negative.")

    if vat_rate is None:
        vat_rate = DEFAULT_VAT_RATE

    if vat_rate < 0 or vat_rate > 1:
        raise ValueError("VAT rate must be between 0 and 1.")

    return round(taxable_amount * vat_rate, 2)


def calculate_total(net_amount: float, vat_amount: float) -> float:
    if net_amount < 0:
        raise ValueError("Net amount cannot be negative.")

    if vat_amount < 0:
        raise ValueError("VAT amount cannot be negative.")

    return round(net_amount + vat_amount, 2)

# ============================================================
# VALIDATION
# ============================================================

def validate_amount(calculated: float, provided: float, tolerance: float = DEFAULT_TOLERANCE) -> dict:
    """
    Compare a calculated value against the value provided
    on the invoice.
    """

    if tolerance < 0:
        raise ValueError("Tolerance cannot be negative.")

    difference = calculated - provided

    return {
        "valid": abs(difference) <= tolerance,
        "calculated": calculated,
        "provided": provided,
        "difference": difference
    }


# ============================================================
# COMPLETE INVOICE CALCULATION
# ============================================================

def calculate_invoice(subtotal: float, discount: float = 0.0, vat_rate: Optional[float] = None) -> dict:
    """
    Perform all invoice calculations.

    Flow:

        Subtotal
            ↓
        Discount
            ↓
        Net Amount
            ↓
        VAT
            ↓
        Total
    """

    if vat_rate is None:
        vat_rate = DEFAULT_VAT_RATE

    net_amount = calculate_net_amount(subtotal=subtotal, discount=discount)

    vat_amount = calculate_vat(taxable_amount=net_amount, vat_rate=vat_rate)

    total = calculate_total(net_amount=net_amount, vat_amount=vat_amount)

    return {
        "subtotal": subtotal,
        "discount": discount,
        "net_amount": net_amount,
        "vat_rate": vat_rate,
        "vat_amount": vat_amount,
        "total": total
    }


# ============================================================
# COMPLETE INVOICE VALIDATION
# ============================================================

def validate_invoice(
    subtotal: float,
    discount: float,
    vat_rate: float,
    provided_vat: float,
    provided_total: float,
    tolerance: float = DEFAULT_TOLERANCE
) -> dict:
    """
    Calculate the expected invoice values and compare them
    against the values extracted from the invoice.
    """

    calculated = calculate_invoice(
        subtotal=subtotal,
        discount=discount,
        vat_rate=vat_rate
    )

    vat_validation = validate_amount(
        calculated=calculated["vat_amount"],
        provided=provided_vat,
        tolerance=tolerance
    )

    total_validation = validate_amount(
        calculated=calculated["total"],
        provided=provided_total,
        tolerance=tolerance
    )

    return {
        "calculation": calculated,
        "vat_validation": vat_validation,
        "total_validation": total_validation,
        "invoice_valid": (
            vat_validation["valid"]
            and total_validation["valid"]
        )
    }

# calculate_agent.py

In [22]:
import ollama

class CalculateAgent:

    def __init__(self, model="qwen2.5:7b"):
        self.model = model

        # ============================================================
        # AVAILABLE PYTHON TOOLS
        # ============================================================

        self.tools = {
            "calculate_net_amount": calculate_net_amount,
            "calculate_vat": calculate_vat,
            "calculate_total": calculate_total,
            "validate_amount": validate_amount,
            "calculate_invoice": calculate_invoice,
            "validate_invoice": validate_invoice,
        }

        # ============================================================
        # OLLAMA TOOL DEFINITIONS
        # ============================================================

        self.tool_definitions = [
            {
                "type": "function",
                "function": {
                    "name": "calculate_net_amount",
                    "description": (
                        "Calculate the amount remaining after subtracting a discount "
                        "from a subtotal. Use ONLY when the user asks for the amount "
                        "after discount."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "subtotal": {
                                "type": "number",
                                "description": "Original amount before discount."
                            },
                            "discount": {
                                "type": "number",
                                "description": "Discount amount."
                            }
                        },
                        "required": ["subtotal", "discount"]
                    }
                }
            },

            {
                "type": "function",
                "function": {
                    "name": "calculate_vat",
                    "description": (
                        "Calculate VAT/tax on a taxable amount. "
                        "Use when the user explicitly asks to calculate VAT or tax. "
                        "vat_rate must be a decimal, for example 14% = 0.14."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "taxable_amount": {
                                "type": "number",
                                "description": "Amount on which VAT is calculated."
                            },
                            "vat_rate": {
                                "type": "number",
                                "description": (
                                    "VAT rate as a decimal. "
                                    "Examples: 14% = 0.14, 5% = 0.05."
                                )
                            }
                        },
                        "required": ["taxable_amount"]
                    }
                }
            },

            {
                "type": "function",
                "function": {
                    "name": "calculate_total",
                    "description": (
                        "Calculate the invoice total by adding a net amount and an already calculated "
                        "VAT amount. Use ONLY when the exact VAT amount is already provided or known. "
                        "Do NOT use this tool when the user provides a VAT percentage/rate instead of "
                        "a VAT amount. If the user provides a net amount and a VAT rate and asks for "
                        "the total, use calculate_invoice instead."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "net_amount": {
                                "type": "number",
                                "description": "Net amount before VAT."
                            },
                            "vat_amount": {
                                "type": "number",
                                "description": "VAT amount."
                            }
                        },
                        "required": ["net_amount", "vat_amount"]
                    }
                }
            },

            {
                "type": "function",
                "function": {
                    "name": "validate_amount",
                    "description": (
                        "Compare an already calculated amount with an amount provided "
                        "by the user. This tool does NOT calculate the expected amount. "
                        "Use only when the calculated/reference amount is already known."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "calculated": {
                                "type": "number",
                                "description": "Already known calculated/reference amount."
                            },
                            "provided": {
                                "type": "number",
                                "description": "Amount provided on the invoice."
                            },
                            "tolerance": {
                                "type": "number",
                                "description": (
                                    "Optional allowed difference. "
                                    "Only include when explicitly provided by the user."
                                )
                            }
                        },
                        "required": ["calculated", "provided"]
                    }
                }
            },

            {
                "type": "function",
                "function": {
                    "name": "calculate_invoice",
                    "description": (
                        "Calculate an invoice total when the user provides an amount and asks to "
                        "calculate VAT and/or the final total. Use this tool when a VAT percentage "
                        "or rate is provided and the VAT amount must be calculated. For example, "
                        "if the user says '8000 before VAT, VAT 14%, calculate the total', use "
                        "this tool. Python performs the VAT and total calculations."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "subtotal": {
                                "type": "number",
                                "description": "Original amount before discount."
                            },
                            "discount": {
                                "type": "number",
                                "description": "Discount amount."
                            },
                            "vat_rate": {
                                "type": "number",
                                "description": (
                                    "Optional VAT rate as a decimal. "
                                    "14% = 0.14. If omitted, Python uses its default."
                                )
                            }
                        },
                        "required": ["subtotal"]
                    }
                }
            },

            {
                "type": "function",
                "function": {
                    "name": "validate_invoice",
                    "description": (
                        "Validate a complete invoice by calculating expected VAT and "
                        "total and comparing them with the VAT and total provided on "
                        "the invoice. Requires subtotal, discount, VAT rate, provided "
                        "VAT, and provided total. Never infer the VAT rate from other values."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "subtotal": {
                                "type": "number",
                                "description": "Original amount before discount."
                            },
                            "discount": {
                                "type": "number",
                                "description": "Discount amount."
                            },
                            "vat_rate": {
                                "type": "number",
                                "description": "VAT rate as a decimal."
                            },
                            "provided_vat": {
                                "type": "number",
                                "description": "VAT amount written on the invoice."
                            },
                            "provided_total": {
                                "type": "number",
                                "description": "Total amount written on the invoice."
                            },
                            "tolerance": {
                                "type": "number",
                                "description": (
                                    "Optional allowed difference. "
                                    "Only include when explicitly provided."
                                )
                            }
                        },
                        "required": [
                            "subtotal",
                            "discount",
                            "vat_rate",
                            "provided_vat",
                            "provided_total"
                        ]
                    }
                }
            }
        ]

    # ============================================================
    # SYSTEM PROMPT
    # ============================================================

    def _system_prompt(self):
        return """
You are the Calculate Agent in an invoice analysis system.

Your ONLY job is to:
1. Understand the user's calculation or validation request.
2. Extract numerical values explicitly stated by the user.
3. Select the correct Python calculation tool.
4. Pass the extracted values to the tool.

Python tools perform ALL financial arithmetic.

============================================================
STRICT RULES
============================================================

1. NEVER perform arithmetic yourself.

Do not calculate:
- VAT
- totals
- discounts
- net amounts
- differences
- percentages
- validation results

Always let the Python tool perform the calculation.

------------------------------------------------------------

2. NEVER INVENT NUMERICAL VALUES.

Only use numbers explicitly provided by the user.

Do NOT infer a missing value from other values.

For example:

User:
"subtotal = 9500 and VAT = 1330. Is the invoice valid?"

Do NOT calculate:
1330 / 9500 = 14%

Do NOT invent vat_rate = 0.14.

If a required value is missing, do not create one.

------------------------------------------------------------

3. DO NOT ADD OPTIONAL ARGUMENTS THAT THE USER DID NOT PROVIDE.

If an optional parameter is not mentioned, OMIT it.

For example, if the user says:

"احسب ضريبة 14% على 10000"

Use:

{
    "taxable_amount": 10000,
    "vat_rate": 0.14
}

Do NOT add:
"tolerance": 0.0

Python will use its own default value when an optional argument is omitted.

------------------------------------------------------------

4. PERCENTAGE CONVERSION

Convert explicitly stated percentages into decimals.

14% = 0.14
5% = 0.05
7.5% = 0.075
10% = 0.10
15% = 0.15

Do not invent a percentage that the user did not provide.

------------------------------------------------------------

5. USE ONLY ONE TOOL CALL.

Select the single most appropriate tool for the user's request.

Do not try to perform multiple tool calls.

If the request requires multiple calculation steps that cannot be
performed by one available tool, do not invent intermediate values.

============================================================
TOOL SELECTION
============================================================

1. calculate_vat

Use when the user asks to calculate VAT/tax for a known taxable amount.

Examples:

"احسب ضريبة 14% على 10000"

"Calculate 5% VAT on 20000"

Use:
calculate_vat

Arguments:
{
    "taxable_amount": 10000,
    "vat_rate": 0.14
}

If the user says "after a discount" and provides both the original
amount and discount, but only asks for the VAT, use calculate_vat
with the resulting taxable amount ONLY if that taxable amount is
explicitly stated by the user.

Do not calculate the discounted amount yourself.

------------------------------------------------------------

2. calculate_net_amount

Use when the user asks for the amount after a discount.

Example:

"احسب صافي 15000 بعد خصم 1000"

Use:
calculate_net_amount

Arguments:
{
    "subtotal": 15000,
    "discount": 1000
}

------------------------------------------------------------

3. calculate_total

Use when the user explicitly provides BOTH:
- net amount
- VAT amount

and asks for the total.

Example:

"صافي المبلغ 10000 والضريبة 1400، احسب الإجمالي"

Use:
calculate_total

Arguments:
{
    "net_amount": 10000,
    "vat_amount": 1400
}

------------------------------------------------------------

4. calculate_invoice

Use when the user asks for the COMPLETE invoice calculation:

subtotal
→ discount
→ net amount
→ VAT
→ total

Example:

"احسب الفاتورة: 20000 قبل الخصم، خصم 2000، وضريبة 14%"

Use:
calculate_invoice

Arguments:
{
    "subtotal": 20000,
    "discount": 2000,
    "vat_rate": 0.14
}

If VAT rate is not explicitly provided, the Python function's default
VAT rate may be used.

------------------------------------------------------------

5. validate_amount

Use when the user wants to compare an existing amount against an
ALREADY KNOWN calculated amount.

This tool does NOT calculate the expected amount.

Example:

"القيمة المحسوبة للضريبة 1400، والفاتورة فيها 1400. هل متساويين؟"

Use:
validate_amount

Arguments:
{
    "calculated": 1400,
    "provided": 1400
}

If tolerance is explicitly provided, include it.

Otherwise OMIT tolerance.

------------------------------------------------------------

6. validate_invoice

Use when the user wants to validate a COMPLETE invoice AND provides
all required values:

- subtotal
- discount
- VAT rate
- provided VAT
- provided total

Example:

"الفاتورة: 10000، خصم 500، ضريبة 14%، الضريبة المكتوبة 1330،
والإجمالي المكتوب 10830. هل صحيحة؟"

Use:
validate_invoice

Arguments:
{
    "subtotal": 10000,
    "discount": 500,
    "vat_rate": 0.14,
    "provided_vat": 1330,
    "provided_total": 10830
}

Do NOT infer the VAT rate from provided VAT.

If VAT rate is missing, do not invent it.

============================================================
MISSING INFORMATION
============================================================

If a required value is missing, do not invent or derive it.

If no appropriate tool can be called safely, do not fabricate a tool call.

============================================================
FINAL RULE
============================================================

The LLM understands the request and selects the tool.

Python performs the mathematics.

Never replace Python calculations with your own arithmetic.
"""

    # ============================================================
    # PARSE USER REQUEST
    # ============================================================

    def parse_request(self, prompt: str):

        response = ollama.chat(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": self._system_prompt()
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            tools=self.tool_definitions
        )

        if not response.message.tool_calls:
            raise ValueError(
                "The model did not select a calculation tool."
            )

        tool_call = response.message.tool_calls[0]

        return {
            "tool": tool_call.function.name,
            "arguments": tool_call.function.arguments
        }

    # ============================================================
    # EXECUTE PYTHON TOOL
    # ============================================================

    def execute_tool(self, tool_name: str, arguments: dict):

        if tool_name not in self.tools:
            raise ValueError(
                f"Unknown calculation tool: {tool_name}"
            )

        # Normalize numeric arguments.
        numeric_fields = {
            "subtotal",
            "discount",
            "taxable_amount",
            "vat_rate",
            "net_amount",
            "vat_amount",
            "calculated",
            "provided",
            "tolerance",
            "provided_vat",
            "provided_total"
        }

        for key in arguments:
            if key in numeric_fields:
                arguments[key] = float(arguments[key])

        tool = self.tools[tool_name]

        return tool(**arguments)

    # ============================================================
    # RUN AGENT
    # ============================================================

    def run(self, prompt: str):

        request = self.parse_request(prompt)

        tool_name = request["tool"]
        arguments = request["arguments"]

        result = self.execute_tool(
            tool_name=tool_name,
            arguments=arguments
        )

        return {
            "success": True,
            "tool": tool_name,
            "arguments": arguments,
            "result": result
        }

In [23]:
import ollama

from typing import Literal
from pydantic import BaseModel, Field, ValidationError
# from calculate_agent import CalculateAgent


# ============================================================
# ROUTE DECISION
# ============================================================

AgentIntent = Literal[
    "calculation_agent",
    "graphical_agent",
    "rag_agent"
]


class RouteDecision(BaseModel):

    reasoning: str = Field(
        default="",
        description="Short reasoning explaining why this agent was selected."
    )
    intent: AgentIntent = Field(
        description="The designated agent destination."
    )

    confidence: float = Field(
        default=1.0,
        ge=0.0,
        le=1.0,
        description="Confidence score between 0.0 and 1.0."
    )


# ============================================================
# ROUTER PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an intent classification and routing engine
for an AI Invoice and Tax Compliance Assistant.

Your ONLY job is to decide which agent should handle
the user's request.

Available agents:

1. 'rag_agent'

Use this when the user asks for information that should
be retrieved from the tax/legal knowledge base, including:

- Egyptian tax laws
- Egyptian VAT laws and regulations
- VAT rates defined by law
- E-invoicing rules and requirements
- Legal requirements
- Tax compliance rules
- Tax regulations
- Explanations based on tax documents
- Questions asking what the law says
- Questions asking about legally defined tax rates
- Questions where the required information must be retrieved
  from the knowledge base

Also use rag_agent when the user asks for a calculation
but the numerical information required to perform the
calculation is missing and the question depends on a
tax law or regulation.

Examples:

"ما هي نسبة ضريبة القيمة المضافة في مصر؟"
→ rag_agent

"ما هي نسبة VAT حسب القانون المصري؟"
→ rag_agent

"اشرح قانون ضريبة القيمة المضافة المصري"
→ rag_agent

"احسب الضريبة حسب قانون ضريبة القيمة المضافة المصري"
→ rag_agent


2. 'calculation_agent'

Use this when the user asks for an actual numerical
calculation AND provides the numerical values needed
for that calculation.

Use this for:

- VAT calculations
- Invoice totals
- Discounts
- Net amounts
- Checking calculated amounts
- Financial calculations
- Arithmetic related to invoices

Examples:

"احسب ضريبة القيمة المضافة على 10000 جنيه بنسبة 14%"
→ calculation_agent

"كم قيمة الضريبة على مبلغ 20000 جنيه بنسبة 14%؟"
→ calculation_agent

"احسب صافي المبلغ بعد خصم 1000 جنيه من 15000 جنيه"
→ calculation_agent

"احسب إجمالي فاتورة قيمتها 10000 جنيه مع ضريبة 14%"
→ calculation_agent


3. 'graphical_agent'

Use this when the user asks for:

- A chart
- A graph
- A plot
- A visualization
- Sales or revenue trends
- VAT trends
- Invoice statistics that require visualization
- Any data visualization


IMPORTANT ROUTING RULES:

1. Do NOT choose calculation_agent only because the
   query contains words such as "calculate", "احسب",
   "كام", or "قيمة".

2. Choose calculation_agent ONLY when the user provides
   the numerical inputs required for an actual calculation.

3. If the user asks to calculate something but the required
   numerical input is missing, do NOT invent or assume a value.

4. If a query asks about Egyptian tax law, VAT regulations,
   legal VAT rates, or information that must come from the
   knowledge base, choose rag_agent.

5. If a query contains both a tax-law question and a numerical
   calculation, choose rag_agent when the calculation depends
   on obtaining legal or regulatory information first.

6. Do NOT infer a VAT rate from the user's wording unless
   the VAT rate is explicitly provided.

7. Do NOT perform calculations yourself.

8. Do NOT answer the user's question.

9. Do NOT extract values for another agent.

10. ONLY decide which agent should handle the query.

11. Return exactly one intent.

The intent MUST be exactly one of:

- calculation_agent
- graphical_agent
- rag_agent
"""


# ============================================================
# ROUTE QUERY
# ============================================================

def route_query(user_query: str) -> dict:

    try:

        response = ollama.chat(
            model="qwen2.5:7b",

            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": f"Analyze and route this query: {user_query}"
                }
            ],

            format="json",

            options={
                "temperature": 0.0
            }
        )

        raw_content = response.message.content

        parsed = RouteDecision.model_validate_json(
            raw_content
        )

        return {
            "intent": parsed.intent,
            "query": user_query,
            "reasoning": parsed.reasoning,
            "confidence": parsed.confidence,
        }

    except ValidationError as ve:

        return {
            "intent": "rag_agent",
            "query": user_query,
            "reasoning": f"Validation Error: {ve}",
            "confidence": 0.0,
        }

    except Exception as e:

        return {
            "intent": "rag_agent",
            "query": user_query,
            "reasoning": f"Ollama Error: {str(e)}",
            "confidence": 0.0,
        }


# ============================================================
# AGENTS
# ============================================================

# These functions are temporary placeholders.
# They will be replaced with the actual agents.

calculate_agent_instance = CalculateAgent(model="qwen2.5:7b")

def rag_agent(query: str, image_path: str = None) -> dict:

    return {
        "text": f"RAG agent received: {query}"
    }


def calculation_agent(prompt: str):

    return calculate_agent_instance.run(prompt)


def graphical_agent(prompt: str) -> str:

    return f"Graphical agent received: {prompt}"


# ============================================================
# ORCHESTRATOR
# ============================================================

def orchestrator(user_query: str, image_path: str = None):

    routing_result = route_query(user_query)

    intent = routing_result["intent"]

    print(
        f"--> Routing request to: {intent} "
        f"(Reason: {routing_result['reasoning']})"
    )

    if intent == "rag_agent":

        return rag_agent(
            query=user_query,
            image_path=image_path
        )

    elif intent == "calculation_agent":

        return calculation_agent(
            user_query
        )

    elif intent == "graphical_agent":

        return graphical_agent(
            user_query
        )

    else:

        return "No suitable agent was identified."


# ============================================================
# TEST
# ============================================================

if __name__ == "__main__":

    test_queries = [

        # ====================================================
        # CALCULATION AGENT
        # ====================================================

        "احسب ضريبة القيمة المضافة على 9500 جنيه بنسبة 14%",

        "احسب إجمالي فاتورة قيمتها 10000 جنيه مع ضريبة 14%",

        "لو السعر 5000 جنيه والخصم 500 جنيه، كام المبلغ النهائي؟",

        "احسب صافي المبلغ بعد خصم 1000 جنيه من 15000 جنيه",

        "كم قيمة الضريبة على مبلغ 20000 جنيه بنسبة 14%؟",

        "الفاتورة قيمتها 12000 جنيه والضريبة 1680 جنيه، هل الحساب صحيح؟",

        "احسب الإجمالي لو المبلغ قبل الضريبة 8000 والضريبة 14%",

        "لو عندي فاتورة بـ 15000 جنيه وخصم 2000 جنيه، احسب الضريبة والإجمالي",

        "هل 1400 جنيه ضريبة صحيحة على مبلغ 10000 جنيه بنسبة 14%؟",

        "احسب قيمة VAT لفاتورة قيمتها 25000 جنيه بنسبة 14%",


        # ====================================================
        # RAG AGENT
        # ====================================================

        "ما هي متطلبات الفاتورة الإلكترونية في مصر؟",

        "ما هي قوانين ضريبة القيمة المضافة في مصر؟",

        "ما هي نسبة ضريبة القيمة المضافة في مصر؟",

        "ما هي البيانات الإلزامية التي يجب أن تحتوي عليها الفاتورة؟",

        "هل الفاتورة الإلكترونية إلزامية للشركات؟",

        "ما هي شروط التسجيل في منظومة الفاتورة الإلكترونية؟",

        "اشرح لي قانون ضريبة القيمة المضافة",

        "ما هي متطلبات إصدار فاتورة إلكترونية صحيحة؟",

        "هل يوجد حد معين للتسجيل في ضريبة القيمة المضافة؟",

        "ما هي القواعد الخاصة بالفواتير الضريبية في مصر؟",


        # ====================================================
        # GRAPHICAL AGENT
        # ====================================================

        "اعمل رسم بياني لإجمالي المبيعات لكل منتج",

        "اعمل chart للمبيعات الشهرية",

        "أريد رسم بياني يوضح الإيرادات خلال السنة",

        "اعمل visualization لعدد الفواتير لكل شهر",

        "اعرض اتجاه المبيعات في رسم بياني",

        "ارسم graph يوضح قيمة الضريبة لكل شهر",

        "اعمل plot للمبيعات حسب المنتج",

        "أريد رسم بياني يقارن إجمالي الفواتير بين الشهور",

        "اعمل chart يوضح توزيع الفواتير حسب المنتج",

        "اعرض بيانات المبيعات في رسم بياني",

        
        # ====================================================
        # BORDERLINE / MIXED
        # ====================================================

        "ما هي نسبة VAT وهل يمكنك حسابها على فاتورة بقيمة 10000؟",

        "اشرح لي ضريبة القيمة المضافة واحسبها على 5000 جنيه",

        "اعمل رسم بياني يوضح نسبة ضريبة القيمة المضافة خلال السنة",

        "هل الفاتورة دي متوافقة مع قانون الفاتورة الإلكترونية؟",

        "احسب الضريبة حسب قانون ضريبة القيمة المضافة المصري",

        "ما هي متطلبات الفاتورة الإلكترونية واحسب لي الإجمالي؟"
    ]


    for i, query in enumerate(test_queries, 1):

        print("\n" + "=" * 70)

        print(f"TEST {i}")

        print("QUERY:")
        print(query)

        result = route_query(query)

        print("\nROUTING RESULT:")
        print(result)


TEST 1
QUERY:
احسب ضريبة القيمة المضافة على 9500 جنيه بنسبة 14%

ROUTING RESULT:
{'intent': 'calculation_agent', 'query': 'احسب ضريبة القيمة المضافة على 9500 جنيه بنسبة 14%', 'reasoning': '', 'confidence': 1.0}

TEST 2
QUERY:
احسب إجمالي فاتورة قيمتها 10000 جنيه مع ضريبة 14%

ROUTING RESULT:
{'intent': 'calculation_agent', 'query': 'احسب إجمالي فاتورة قيمتها 10000 جنيه مع ضريبة 14%', 'reasoning': '', 'confidence': 1.0}

TEST 3
QUERY:
لو السعر 5000 جنيه والخصم 500 جنيه، كام المبلغ النهائي؟

ROUTING RESULT:
{'intent': 'calculation_agent', 'query': 'لو السعر 5000 جنيه والخصم 500 جنيه، كام المبلغ النهائي؟', 'reasoning': '', 'confidence': 1.0}

TEST 4
QUERY:
احسب صافي المبلغ بعد خصم 1000 جنيه من 15000 جنيه

ROUTING RESULT:
{'intent': 'calculation_agent', 'query': 'احسب صافي المبلغ بعد خصم 1000 جنيه من 15000 جنيه', 'reasoning': '', 'confidence': 1.0}

TEST 5
QUERY:
كم قيمة الضريبة على مبلغ 20000 جنيه بنسبة 14%؟

ROUTING RESULT:
{'intent': 'calculation_agent', 'query': 'كم قيمة الضريبة على م

In [15]:
result = orchestrator(
    "احسب ضريبة القيمة المضافة على 9500 جنيه بنسبة 14%"
)

print(result)

--> Routing request to: calculation_agent (Reason: )
{'success': True, 'tool': 'calculate_vat', 'arguments': {'taxable_amount': 9500.0, 'vat_rate': 0.14}, 'result': 1330.0}
